In [1]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import TensorDataset, random_split
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_dataset
from core.data.dataset import create_dataloaders
from core.data.transforms import create_normalizer_from_data
from core.model.bert import BertForPretraining
from core.training.sampler import create_kde_sampler
from core.training.pretrainer import setup_training
from core.logger import print_data_summary, log_model_summary

In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BERT Configuration")

PyTorch version: 2.9.0+cu126
Using device: cuda
Working directory: /home/jessiez/projects/osu_corpora

--- Loaded BERT Configuration ---
data:
  max_seq_len: 2048
  val_split: 0.1
  max_samples_per_class:
    aim: 2500
    tech: 2500
  min_stars: 4.0
  max_stars: 12.0
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
  local_attention_window: 256
  cnn_kernel_size: 15
components:
  use_flash_attention: true
  compile_model: true
  compile_mode: default
pretraining:
  db_path: ./data/beatmap_dataset_test/
  batch_size: 8
  num_epochs: 8
  learning_rate: 0.0002
  min_lr: 1.0e-06
  cooldown_type: cosine
  weight_decay: 0.05
  warmup_ratio: 0.1
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./checkpoints
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 8
  difficulty_loss_weight: 1.0
  mlm_loss_weight: 1.0
  masking_ratio: 0.3
  mean_span_length: 4
  sampling:
    method: kde
    kde_bandwidth: 0.2
    num_bins: 200
finetuning:
  db_path: 

In [3]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['pretraining']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_attributes, loaded_ids = load_dataset(
    DATASET_PATH, 
    max_seq_len=config['data']['max_seq_len'],
    raw_beatmap_path=config['pretraining'].get('raw_beatmap_path', './data/raw')
)

print_data_summary(all_beatmaps_data)

Using database: ./data/beatmap_dataset_test/
Loading raw data from Parquet dataset...
Loading beatmap metadata...
Found metadata for 19656 beatmaps. Processing in chunks of 5000...


Processing Chunks: 100%|██████████| 4/4 [00:22<00:00,  5.68s/it]


Consolidating processed chunks...
Loaded raw feature vectors for 19623 beatmaps.
Calculating difficulty attributes (will use cache if available)...


Calculating Attributes: 100%|██████████| 19/19 [00:00<00:00, 160668.90it/s]



--- Data Summary ---
Total beatmaps: 19604
Vector dimension: 18
Sequence length - Min: 37, Max: 2048, Avg: 828.0
--------------------


In [4]:
val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size

temp_dataset = TensorDataset(torch.arange(len(all_beatmaps_data)))
train_split, val_split = random_split(temp_dataset, [train_size, val_size])

print(f"Data split: {len(train_split.indices)} training, {len(val_split.indices)} validation")

train_data_list = [all_beatmaps_data[i] for i in train_split.indices]
val_data_list = [all_beatmaps_data[i] for i in val_split.indices]

train_attributes = {key: val[train_split.indices] for key, val in difficulty_attributes.items()}
val_attributes = {key: val[val_split.indices] for key, val in difficulty_attributes.items()}

sampler = create_kde_sampler(
    train_attributes['stars'],
    bandwidth=config['pretraining']['sampling']['kde_bandwidth'],
    num_bins=config['pretraining']['sampling'].get('num_bins', 100),
)

normalizer = create_normalizer_from_data(train_data_list, train_attributes)
vector_stats = normalizer.get_vector_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")

Data split: 17644 training, 1960 validation
Creating optimized KDE sampler with bandwidth=0.2, bins=200...
KDE sampling - Min weight: 0.1698, Max weight: 705.1794
Calculating normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
------------------------------------------------------------
Field Name             Type         Param 1      Param 2     
------------------------------------------------------------
norm_x                 none         N/A          N/A         
norm_y                 none         N/A          N/A         
delta_x                mean/std     -0.0008      110.1833    
delta_y                mean/std     0.0028       98.7382     
log_time_diff_ms       mean/std     4.8676       0.4986      
bpm                    mean/std     184.8389     37.4357     
notes_per_second       mean/std     8.1564       2.7623      
velocity               mean/std     0.8951       0.8925      
relative_angle         none         N/A        

In [5]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    train_attributes,
    val_attributes,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['pretraining']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}")
print(f"Sample attributes keys: {list(sample_batch[2].keys())}")

Created dataloaders with batch size: 8
Sample batch shapes: vectors=torch.Size([8, 2048, 18]), mask=torch.Size([8, 2048])
Sample attributes keys: ['stars', 'aim', 'speed', 'slider_factor', 'cs', 'ar', 'slider_multiplier']


In [6]:
model = BertForPretraining.from_config(config, device)

log_model_summary(model)

print("\nRunning a test forward pass with mixed precision (autocast)...")
use_amp_for_test = device.type == "cuda"
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp_for_test):
        sample_vectors, sample_mask, sample_attrs, sample_cu_seqlens = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)

        predictions, targets, _ = model(sample_vectors, sample_mask)

        print(f"Prediction output keys: {list(predictions.keys())}")
        print(f"MLM prediction keys: {list(predictions['mlm'].keys())}")
        print(f"Difficulty prediction keys: {list(predictions['difficulty'].keys())}")

print("\nBERT model created and tested successfully!")

Compiling BERT pre-training model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 33.30M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
Flash Attention: True
------------------------------

--- Pre-training Head Information ---
Tasks: Masked Modeling, Difficulty Attribute Prediction
Masking Ratio: 0.3
Model Compiled: True
------------------------------

Running a test forward pass with mixed precision (autocast)...


/home/jessiez/projects/osu_corpora/.venv/lib/python3.12/site-packages/torch/_dynamo/utils.py:3546: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return node.target(*args, **kwargs)  # type: ignore[operator]
W1225 17:48:55.254000 500364 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1558] [8/0_1] Not enough SMs to use max_autotune_gemm mode
/home/jessiez/projects/osu_corpora/.venv/lib/python3.12/site-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch

Prediction output keys: ['mlm', 'difficulty']
MLM prediction keys: ['continuous', 'categorical']
Difficulty prediction keys: ['stars', 'aim', 'speed', 'slider_factor', 'cs', 'ar', 'slider_multiplier']

BERT model created and tested successfully!


In [7]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device, normalizer
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        loaded_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch = loaded_epoch + 1 
        print(f"Loaded checkpoint from epoch {loaded_epoch}, resuming from epoch {start_epoch}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting pretraining from scratch")

print(f"Pretraining setup complete. Starting from epoch {start_epoch}")
print(f"Total epochs: {config['pretraining']['num_epochs']}")

Scheduler: WSD with 220 warmup, 220 stable, 1768 decay steps.
Cooldown type: cosine, Min LR Ratio: 0.0050
PreTrainer initialized - AMP: True, Device: cuda, Grad Accum: 8
Effective batch size: 64
Pretraining setup complete. Starting from epoch 0
Total epochs: 8


In [ ]:
print("\nStarting BERT pretraining...")
print(f"BERT Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")

print(f"Pretraining samples: {len(train_data_list)} base maps")
print(f"Validation samples: {len(val_data_list)} base maps")

metrics_tracker = trainer.train(start_epoch)

print("\nBERT training completed!")


Starting BERT pretraining...
BERT Model: 6 layers, 512 dimensions
Pretraining samples: 17644 base maps
Validation samples: 1960 base maps

--- Starting Training ---
Epochs: 1 to 8
Batch Size: 8
Learning Rate: 0.0002
------------------------------


Epoch 1 [Train]:   0%|          | 0/276 [00:00<?, ?it/s]

Epoch 1 [Validate]:   0%|          | 0/245 [00:00<?, ?it/s]

Epoch 1/8 | Train Loss: 36.0747 (MLM: 9.9655, Diff: 26.1091) | Val Loss: 6.8982 (MLM: 6.2525, Diff: 0.6457) | LR: 2.00e-04 | Time: 339.98s
------------------------------
Validation Metrics
Difficulty MAE:
  aim            : 0.2919
  ar             : 0.2298
  cs             : 0.3087
  slider_factor  : 0.0390
  slider_multiplier: 0.2596
  speed          : 0.1767
  stars          : 0.4749
Continuous Features:
  norm_x         : MAE 0.2634
  norm_y         : MAE 0.2954
  delta_x        : MAE 0.6533
  delta_y        : MAE 0.6608
  log_time_diff_ms: MAE 0.5136
  bpm            : MAE 0.2140
  notes_per_second: MAE 0.3335
  velocity       : MAE 0.4530
  relative_angle : MAE 0.6636
  rhythm_change  : MAE 0.1310
  log_slider_pixel_length: MAE 0.7518
  slider_repeats : MAE 0.1337
  slider_tortuosity: MAE 0.1878
Categorical Features:
  object_type    : Acc 68.50%, Prec 0.6846, Rec 0.6850
  is_new_combo   : Acc 81.99%, Prec 0.7888, Rec 0.8199
  beat_in_measure: Acc 63.53%, Prec 0.6389, Rec 0.6353
 

Epoch 2 [Train]:   0%|          | 0/276 [00:00<?, ?it/s]